In [1]:
from pathlib import Path

# Thiết lập đường dẫn tương đối tới thư mục dữ liệu gốc
raw_dir = Path("data/raw")

print("=== KIỂM TRA CẤU TRÚC THƯ MỤC VÀ TÊN FILE TRONG DATA/RAW ===")

if not raw_dir.exists():
    print(f"Lỗi: Không tìm thấy thư mục {raw_dir.resolve()}. Hãy kiểm tra lại vị trí đặt dữ liệu!")
else:
    # Lấy danh sách tất cả các file ảnh bên trong data/raw
    image_extensions = {".jpg", ".jpeg", ".png"}
    all_images = [p for p in raw_dir.glob("**/*") if p.suffix.lower() in image_extensions]
    
    print(f"Tổng số lượng ảnh tìm thấy: {len(all_images)}")
    print("\n--- 5 Đường dẫn mẫu đầu tiên (để xem cách phân cấp thư mục và tên file) ---")
    for img in all_images[:5]:
        # In ra đường dẫn tương đối so với thư mục raw để quan sát
        print(img.relative_to(raw_dir))

=== KIỂM TRA CẤU TRÚC THƯ MỤC VÀ TÊN FILE TRONG DATA/RAW ===
Lỗi: Không tìm thấy thư mục D:\Project\notebook\data\raw. Hãy kiểm tra lại vị trí đặt dữ liệu!


In [2]:
import os
from pathlib import Path

# Đưa thư mục làm việc về thư mục gốc của project (nếu đang đứng ở thư mục notebook)
if os.path.basename(os.getcwd()) == "notebook" or os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Thư mục làm việc hiện tại:", os.getcwd())

# Thiết lập đường dẫn tới thư mục raw
raw_dir = Path("data/raw")

print("\n=== KIỂM TRA CẤU TRÚC THƯ MỤC VÀ TÊN FILE TRONG DATA/RAW ===")

if not raw_dir.exists():
    print(f"Lỗi: Vẫn không tìm thấy thư mục {raw_dir.resolve()}. Hãy kiểm tra xem thư mục 'data/raw' có nằm trong D:\\Project không nhé!")
else:
    image_extensions = {".jpg", ".jpeg", ".png"}
    all_images = [p for p in raw_dir.glob("**/*") if p.suffix.lower() in image_extensions]
    
    print(f"Tổng số lượng ảnh tìm thấy: {len(all_images)}")
    print("\n--- 5 Đường dẫn mẫu đầu tiên để xem cấu trúc ---")
    for img in all_images[:5]:
        print(img.relative_to(raw_dir))

Thư mục làm việc hiện tại: D:\Project

=== KIỂM TRA CẤU TRÚC THƯ MỤC VÀ TÊN FILE TRONG DATA/RAW ===
Tổng số lượng ảnh tìm thấy: 2137

--- 5 Đường dẫn mẫu đầu tiên để xem cấu trúc ---
Healthy Fish\Healthy Fish\Silver_carp_(Hypophthalmichthys molitrix)\Fish_83\IMG20260315103549.jpg
Healthy Fish\Healthy Fish\Silver_carp_(Hypophthalmichthys molitrix)\Fish_83\IMG20260315103638.jpg
Healthy Fish\Healthy Fish\Silver_carp_(Hypophthalmichthys molitrix)\Fish_83\IMG20260315103656.jpg
Healthy Fish\Healthy Fish\Silver_carp_(Hypophthalmichthys molitrix)\Fish_83\IMG20260315103704.jpg
Healthy Fish\Healthy Fish\Silver_carp_(Hypophthalmichthys molitrix)\Fish_83\IMG20260315103705.jpg


In [5]:
import os
import cv2
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
import imagehash
from tqdm import tqdm
import random

# Đảm bảo đang đứng ở thư mục gốc Project
if os.path.basename(os.getcwd()) == "notebook" or os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

RAW_DIR = Path("data/raw")
DATA_DIR = Path("data")  # Thư mục chính chứa file metadata
SPLITS_DIR = Path("data/splits")

DATA_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

image_extensions = {".jpg", ".jpeg", ".png"}
data_records = []

# ==========================================
# BƯỚC 1: QUÉT RAW IMAGES & KIỂM TRA ẢNH HỎNG
# ==========================================
print("=== BƯỚC 1: Quét và kiểm tra ảnh hỏng từ thư mục raw ===")
for class_folder in RAW_DIR.iterdir():
    if class_folder.is_dir():
        class_name = class_folder.name  # Tên lớp bệnh
        
        for img_path in class_folder.glob("**/*"):
            if img_path.suffix.lower() in image_extensions:
                filename = img_path.name
                specimen_id = img_path.parent.name
                
                if specimen_id == class_name or specimen_id == "":
                    specimen_id = filename.split('_')[0]

                # Kiểm tra ảnh hỏng / lỗi đọc
                try:
                    with Image.open(img_path) as img:
                        img.verify()
                    temp_img = cv2.imread(str(img_path))
                    if temp_img is None:
                        continue
                    data_records.append({
                        "image_path": str(img_path.relative_to(Path(".")) ),
                        "filename": filename,
                        "class_name": class_name,
                        "specimen_id": specimen_id,
                        "abs_path": str(img_path)
                    })
                except Exception:
                    # Bỏ qua ảnh hỏng
                    pass

df_raw = pd.DataFrame(data_records)
print(f"Tổng số ảnh hợp lệ ban đầu: {len(df_raw)}")

# ==========================================
# BƯỚC 2: KIỂM TRA & LOẠI TRÙNG LẶP BẰNG pHASH
# ==========================================
print("\n=== BƯỚC 2: Kiểm tra loại ảnh trùng lặp bằng pHash ===")
hashes = {}
cleaned_records = []
duplicate_records = []

for idx, row in tqdm(df_raw.iterrows(), total=len(df_raw), desc="Calculating pHash"):
    try:
        pil_img = Image.open(row['abs_path'])
        phash = imagehash.phash(pil_img)
        
        is_dup = False
        for existing_hash, existing_row in hashes.items():
            if phash - existing_hash <= 5:  # Ngưỡng pHash
                is_dup = True
                duplicate_records.append({
                    'image_path': row['image_path'],
                    'duplicate_of': existing_row['image_path'],
                    'phash_distance': phash - existing_hash
                })
                break
                
        if not is_dup:
            hashes[phash] = row
            cleaned_records.append({
                "image_path": row["image_path"],
                "filename": row["filename"],
                "class_name": row["class_name"],
                "specimen_id": row["specimen_id"]
            })
    except Exception as e:
        print(f"Lỗi xử lý ảnh {row['image_path']}: {e}")

# Lưu file kiểm toán trùng lặp để làm minh chứng
df_duplicates = pd.DataFrame(duplicate_records)
df_duplicates.to_csv(DATA_DIR / "removed_duplicates.csv", index=False)
print(f"Đã lưu file kiểm toán: data/removed_duplicates.csv (Số ảnh trùng bị loại: {len(df_duplicates)})")

# Tạo DataFrame metadata sạch và lưu vào metadata_clean.csv
df_meta = pd.DataFrame(cleaned_records)
metadata_path = DATA_DIR / "metadata_clean.csv"
df_meta.to_csv(metadata_path, index=False)
print(f"Đã lưu metadata sạch: {metadata_path} (Tổng số ảnh sau dedup: {len(df_meta)})")

# ==========================================
# BƯỚC 3: CHIA TẬP GROUP-AWARE + CLASS-AWARE (70/15/15)
# ==========================================
print("\n=== BƯỚC 3: Chia tập Train/Val/Test (Group-aware + Class-aware) ===")
random.seed(42)
train_rows = []
val_rows = []
test_rows = []

# Chia độc lập theo từng lớp bệnh để đảm bảo cân bằng phân bố lớp
for class_name, group in df_meta.groupby('class_name'):
    unique_specimens = group['specimen_id'].unique().tolist()
    random.shuffle(unique_specimens)
    
    n_specs = len(unique_specimens)
    n_train = int(n_specs * 0.70)
    n_val = int(n_specs * 0.15)
    
    # Đảm bảo mỗi tập có ít nhất số lượng tối thiểu nếu lớp có ít specimen
    if n_specs >= 3:
        if n_train == 0: n_train = 1
        if n_val == 0: n_val = 1
        if n_train + n_val >= n_specs:
            n_train = n_specs - 2
            n_val = 1
            
    train_specs = set(unique_specimens[:n_train])
    val_specs = set(unique_specimens[n_train:n_train + n_val])
    test_specs = set(unique_specimens[n_train + n_val:])
    
    for _, row in group.iterrows():
        row_dict = row.to_dict()
        if row['specimen_id'] in train_specs:
            train_rows.append(row_dict)
        elif row['specimen_id'] in val_specs:
            val_rows.append(row_dict)
        elif row['specimen_id'] in test_specs:
            test_rows.append(row_dict)

df_train = pd.DataFrame(train_rows)
df_val = pd.DataFrame(val_rows)
df_test = pd.DataFrame(test_rows)

# Kiểm tra điều kiện bắt buộc giao nhau bằng 0 giữa các specimen
train_specs_set = set(df_train['specimen_id'])
val_specs_set = set(df_val['specimen_id'])
test_specs_set = set(df_test['specimen_id'])

assert len(train_specs_set.intersection(val_specs_set)) == 0, "Lỗi: Train và Val bị trùng specimen!"
assert len(train_specs_set.intersection(test_specs_set)) == 0, "Lỗi: Train và Test bị trùng specimen!"
assert len(val_specs_set.intersection(test_specs_set)) == 0, "Lỗi: Val và Test bị trùng specimen!"
print("Kiểm tra độc lập Data Leakage: THÀNH CÔNG (Các tập hoàn toàn tách biệt cá thể, giao nhau = 0).")

# Lưu các file phân chia vào data/splits/
df_train.to_csv(SPLITS_DIR / "train.csv", index=False)
df_val.to_csv(SPLITS_DIR / "val.csv", index=False)
df_test.to_csv(SPLITS_DIR / "test.csv", index=False)

# ==========================================
# BƯỚC 4: LẬP BẢNG THỐNG KÊ KẾT QUẢ THEO YÊU CẦU
# ==========================================
print("\n=== BƯỚC 4: Bảng thống kê chi tiết kết quả phân chia ===")
summary_data = []

for class_name, group in df_meta.groupby('class_name'):
    total_images = len(group)
    total_specs = group['specimen_id'].nunique()
    
    sub_train = group[group['specimen_id'].isin(train_specs_set)]
    sub_val = group[group['specimen_id'].isin(val_specs_set)]
    sub_test = group[group['specimen_id'].isin(test_specs_set)]
    
    summary_data.append({
        'Disease': class_name,
        'Total Images': total_images,
        'Total Specimens': total_specs,
        'Train Specimens': sub_train['specimen_id'].nunique(),
        'Val Specimens': sub_val['specimen_id'].nunique(),
        'Test Specimens': sub_test['specimen_id'].nunique(),
        'Train Images': len(sub_train),
        'Val Images': len(sub_val),
        'Test Images': len(sub_test)
    })

df_summary = pd.DataFrame(summary_data)

# Thêm dòng TOTAL
total_row = {
    'Disease': 'TOTAL',
    'Total Images': df_summary['Total Images'].sum(),
    'Total Specimens': df_meta['specimen_id'].nunique(),
    'Train Specimens': df_summary['Train Specimens'].sum(),
    'Val Specimens': df_summary['Val Specimens'].sum(),
    'Test Specimens': df_summary['Test Specimens'].sum(),
    'Train Images': df_summary['Train Images'].sum(),
    'Val Images': df_summary['Val Images'].sum(),
    'Test Images': df_summary['Test Images'].sum()
}
df_summary = pd.concat([df_summary, pd.DataFrame([total_row])], ignore_index=True)

print(df_summary.to_string(index=False))


=== BƯỚC 1: Quét và kiểm tra ảnh hỏng từ thư mục raw ===
Tổng số ảnh hợp lệ ban đầu: 2137

=== BƯỚC 2: Kiểm tra loại ảnh trùng lặp bằng pHash ===


Calculating pHash: 100%|██████████| 2137/2137 [05:44<00:00,  6.20it/s]


Đã lưu file kiểm toán: data/removed_duplicates.csv (Số ảnh trùng bị loại: 936)
Đã lưu metadata sạch: data\metadata_clean.csv (Tổng số ảnh sau dedup: 1201)

=== BƯỚC 3: Chia tập Train/Val/Test (Group-aware + Class-aware) ===
Kiểm tra độc lập Data Leakage: THÀNH CÔNG (Các tập hoàn toàn tách biệt cá thể, giao nhau = 0).

=== BƯỚC 4: Bảng thống kê chi tiết kết quả phân chia ===
                            Disease  Total Images  Total Specimens  Train Specimens  Val Specimens  Test Specimens  Train Images  Val Images  Test Images
            Bacterial Gill Diseases           157               27               18              4               5           107          19           31
             Bacterial Red Diseases           377               35               24              5               6           247          47           83
Epizootic Ulcerative Syndrome (EUS)           177               14                9              2               3           121          21           35
       

In [6]:
import pandas as pd
from pathlib import Path

# Đọc các file split đã chia theo specimen_id chuẩn
df_train = pd.read_csv("data/splits/train.csv")
df_val = pd.read_csv("data/splits/val.csv")
df_test = pd.read_csv("data/splits/test.csv")

# Lấy tập hợp các specimen_id của từng tập
train_specs = set(df_train['specimen_id'])
val_specs = set(df_val['specimen_id'])
test_specs = set(df_test['specimen_id'])

# Kiểm tra giao nhau (Intersection)
train_val_overlap = train_specs.intersection(val_specs)
train_test_overlap = train_specs.intersection(test_specs)
val_test_overlap = val_specs.intersection(test_specs)

print("=== CHỨNG MINH KHÔNG CÓ DATA LEAKAGE GIỮA CÁC TẬP ===")
print(f"Số lượng specimen tập Train: {len(train_specs)}")
print(f"Số lượng specimen tập Val:   {len(val_specs)}")
print(f"Số lượng specimen tập Test:  {len(test_specs)}")
print("-" * 50)
print(f"Giao giữa Train và Val  (Train ∩ Val):  {len(train_val_overlap)} (Phải là 0)")
print(f"Giao giữa Train và Test (Train ∩ Test): {len(train_test_overlap)} (Phải là 0)")
print(f"Giao giữa Val và Test   (Val ∩ Test):   {len(val_test_overlap)} (Phải là 0)")

if len(train_val_overlap) == 0 and len(train_test_overlap) == 0 and len(val_test_overlap) == 0:
    print("\n>>> KẾT LUẬN: TUYỆT ĐỐI KHÔNG CÓ SỰ TRÙNG LẶP CÁ THỂ (SPECIMEN) GIỮA CÁC TẬP. ĐẠT CHUẨN ĐỒ ÁN!")
else:
    print("\n>>> CẢNH BÁO: VẪN CÒN SỰ TRÙNG LẶP!")

=== CHỨNG MINH KHÔNG CÓ DATA LEAKAGE GIỮA CÁC TẬP ===
Số lượng specimen tập Train: 58
Số lượng specimen tập Val:   12
Số lượng specimen tập Test:  16
--------------------------------------------------
Giao giữa Train và Val  (Train ∩ Val):  0 (Phải là 0)
Giao giữa Train và Test (Train ∩ Test): 0 (Phải là 0)
Giao giữa Val và Test   (Val ∩ Test):   0 (Phải là 0)

>>> KẾT LUẬN: TUYỆT ĐỐI KHÔNG CÓ SỰ TRÙNG LẶP CÁ THỂ (SPECIMEN) GIỮA CÁC TẬP. ĐẠT CHUẨN ĐỒ ÁN!


In [7]:
import pandas as pd

df_meta = pd.read_csv("data/metadata_clean.csv")
df_train = pd.read_csv("data/splits/train.csv")
df_val = pd.read_csv("data/splits/val.csv")
df_test = pd.read_csv("data/splits/test.csv")

summary = pd.DataFrame({
    'Total Images': df_meta.groupby('class_name')['filename'].count(),
    'Total Specimens': df_meta.groupby('class_name')['specimen_id'].nunique(),
    'Train Images': df_train.groupby('class_name')['filename'].count(),
    'Val Images': df_val.groupby('class_name')['filename'].count(),
    'Test Images': df_test.groupby('class_name')['filename'].count()
}).fillna(0).astype(int)

print(summary)

                                     Total Images  Total Specimens  \
class_name                                                           
Bacterial Gill Diseases                       157               27   
Bacterial Red Diseases                        377               35   
Epizootic Ulcerative Syndrome (EUS)           177               14   
Healthy Fish                                  490               10   

                                     Train Images  Val Images  Test Images  
class_name                                                                  
Bacterial Gill Diseases                       107          19           31  
Bacterial Red Diseases                        247          47           83  
Epizootic Ulcerative Syndrome (EUS)           121          21           35  
Healthy Fish                                  388          26           76  
